# NeurIPS 2026 replication, notebook 4: Paper A drift trajectories (GPU)
Fine-tunes ViT-B/16-IN21K with its head over the 10 CIFAR-100 stages at three backbone learning rates, 1000 iterations per stage (batch 128), measuring gamma_min(phi_t) every 200 iterations and at each stage end, and the class-IL accuracy of the fine-tuned model against a head trained the same way on the frozen features. About 70 min per (lr, seed) on a T4; 3 lrs x 5 seeds is about 17 GPU-hours, so run it over several sessions: the JSON is saved after every (lr, seed) run and a rerun skips finished pairs (the cell copies the Drive copy back first). To halve the cost, set SEEDS = '0,1,2' first and add seeds later.

In [ ]:
# ---- 0. Setup: Drive, repo, packages ---------------------------------------------
# Run this cell first in every notebook. Session state is lost between sessions;
# the feature caches on Drive are not.
from google.colab import drive; drive.mount('/content/drive')
import os, subprocess, sys, pathlib
WORK = pathlib.Path('/content/drive/MyDrive/neurips2026_replication'); WORK.mkdir(parents=True, exist_ok=True)
REPO_ZIP = WORK / 'repo.zip'            # upload the repo zip here once (claude_replication/repo.zip), or set GIT_URL
GIT_URL = ''                            # e.g. 'https://github.com/<owner>/<repo>.git' once the repo is public
if GIT_URL:
    subprocess.run(['git','clone','--depth','1',GIT_URL,'/content/repo'], check=True)
else:
    assert REPO_ZIP.exists(), f'upload the repo zip to {REPO_ZIP}'
    subprocess.run(['rm','-rf','/content/repo','/content/repo_unz']); subprocess.run(['unzip','-q','-o',str(REPO_ZIP),'-d','/content/repo_unz'], check=True)
    inner = next(p for p in pathlib.Path('/content/repo_unz').rglob('replicate.py')).parent
    subprocess.run(['mv', str(inner), '/content/repo'], check=True)
os.chdir('/content/repo'); sys.path.insert(0, '/content/repo')
os.environ['REPLICATION_FEATURES'] = str(WORK / 'features')
os.environ['REPLICATION_ALLOW_DOWNLOADS'] = '1'   # this notebook runs on Colab only; the subprocess below inherits the download permission (pipeline/features.py::_assert_download_allowed)
subprocess.run([sys.executable,'-m','pip','install','-q','transformers>=4.44','datasets','gdown','scipy','scikit-learn'], check=True)
import torch; print('torch', torch.__version__, 'cuda', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
print('features ->', os.environ['REPLICATION_FEATURES'])

In [ ]:
import subprocess, sys, shutil, pathlib
SEEDS = '0,1,2,3,4'; LRS = '1e-5,1e-4,1e-3'; ITERS_PER_STAGE = 1000; MEASURE_EVERY = 200; GAMMA_SUBSAMPLE = 50   # images/class for the gamma pass; 0 = all 50k
prev = WORK / 'results' / 'paperA_drift.json'
if prev.exists(): pathlib.Path('results').mkdir(exist_ok=True); shutil.copy(prev, 'results/paperA_drift.json')   # resume
subprocess.run([sys.executable, 'replicate.py', 'drift', '--data-root', '/content/data', '--seeds', SEEDS, '--lrs', LRS,
                '--iters-per-stage', str(ITERS_PER_STAGE), '--measure-every', str(MEASURE_EVERY), '--gamma-subsample', str(GAMMA_SUBSAMPLE)], check=True)
(WORK / 'results').mkdir(exist_ok=True); shutil.copy('results/paperA_drift.json', prev)
import json; print(json.dumps(json.load(open('results/paperA_drift.json'))['summary'], indent=1))
# the drift figure (gamma_min trajectories and the accuracy crossover) with its .data.json sidecar
subprocess.run([sys.executable, 'replicate.py', 'figures'], check=True)
shutil.copytree('results/figures', WORK / 'results' / 'figures_drift', dirs_exist_ok=True)
